# Deep Learning ile SNR Tahmini — Colab Runner

Bu notebook sadece **çalıştırma ortamı**. Kodun asıl yeri repodaki `src/` klasörü.

Kod GitHub'da güncellendiyse: **1. hücreyi tekrar çalıştır**, sonra **Runtime → Restart session** yapıp hücreleri baştan çalıştır.

GPU için: **Runtime → Change runtime type → T4 GPU**

## 1. Repoyu çek ve kur

In [ ]:
import os

REPO_URL = "https://github.com/rftbrkzkn/snr-estimation-dl"
REPO_DIR = "/content/snr-estimation-dl"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "yok (CPU)")

---
## 2. Veri üretimi + constellation diyagramı

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.data_gen import generate_frames
from src.plots import plot_constellations

rng = np.random.default_rng(0)
snrs = [20, 10, 0, -5]
frames = [generate_frames(1, 2000, snr_db=s, rng=rng)[0][0] for s in snrs]

plot_constellations(frames, snrs)
plt.show()

---
## 3. M2M4 baseline — RMSE vs SNR

In [ ]:
from src.data_gen import generate_test_set
from src.classical import m2m4
from src.evaluate import per_snr_metrics
from src.plots import plot_metric_vs_snr

X_test, y_test = generate_test_set()          # 16 SNR noktası x 1000 frame
y_m2m4, valid = m2m4(X_test, return_valid=True)

df_m2m4 = per_snr_metrics(y_test, y_m2m4, valid)
display(df_m2m4.round(3))

plot_metric_vs_snr({"M2M4": df_m2m4}, "rmse")
plot_metric_vs_snr({"M2M4": df_m2m4}, "bias")
plt.show()

---
## 4. Model eğitimi

GPU açık olmalı (**Runtime → Change runtime type → T4 GPU**).

Her frame modele verilmeden önce **birim güce normalize** edilir (alıcıdaki AGC gibi). Böylece model SNR'ı toplam güçten "kopya çekerek" öğrenemez.

### 4.1 Eğitim ve doğrulama verisi

In [ ]:
from src import config
from src.data_gen import generate_frames, to_iq
from src.features import extract_features, Standardizer
from src.train import set_seed

set_seed()
N = config.FRAME_LEN
rng = np.random.default_rng(config.SEED)

r_train, y_train = generate_frames(config.N_TRAIN, N, rng=rng)
r_val,   y_val   = generate_frames(config.N_VAL,   N, rng=rng)
y_train, y_val = y_train.astype(np.float32), y_val.astype(np.float32)

# Ham I/Q girdisi (MLP ve 1D CNN için)
X_train_iq, X_val_iq, X_test_iq = to_iq(r_train), to_iq(r_val), to_iq(X_test)

# Öznitelik girdisi (Feature + MLP için)
scaler = Standardizer().fit(extract_features(r_train))
F_train = scaler.transform(extract_features(r_train))
F_val   = scaler.transform(extract_features(r_val))
F_test  = scaler.transform(extract_features(X_test))

print("I/Q:", X_train_iq.shape, " öznitelik:", F_train.shape)

### 4.2 Feature + MLP

In [ ]:
from src.models import FeatureMLP, RawMLP, CNN1D, count_params
from src.train import train_model, predict

models, histories, preds, infer_ms = {}, {}, {"M2M4": y_m2m4}, {}

name = "Feature + MLP"
models[name] = FeatureMLP(F_train.shape[1])
histories[name] = train_model(models[name], F_train, y_train, F_val, y_val)
preds[name], infer_ms[name] = predict(models[name], F_test)

### 4.3 MLP (ham I/Q)

In [ ]:
name = "MLP"
models[name] = RawMLP(N)
histories[name] = train_model(models[name], X_train_iq, y_train, X_val_iq, y_val)
preds[name], infer_ms[name] = predict(models[name], X_test_iq)

### 4.4 1D CNN

In [ ]:
name = "1D CNN"
models[name] = CNN1D()
histories[name] = train_model(models[name], X_train_iq, y_train, X_val_iq, y_val)
preds[name], infer_ms[name] = predict(models[name], X_test_iq)

---
## 5. Değerlendirme ve bildiri figürleri

In [ ]:
import os, time
import pandas as pd
from src.plots import plot_training_curves, plot_pred_vs_true

results = {m: per_snr_metrics(y_test, p) for m, p in preds.items()}
results["M2M4"] = df_m2m4

plot_metric_vs_snr(results, "rmse")
plot_metric_vs_snr(results, "bias")
plot_training_curves(histories)
plot_pred_vs_true(y_test, preds)
plt.show()

In [ ]:
# M2M4 inference süresi (CPU, numpy)
t0 = time.time(); m2m4(X_test); infer_ms["M2M4"] = (time.time() - t0) * 1000 / len(X_test)

low = y_test <= -4      # M2M4'ün çöktüğü bölge
summary = pd.DataFrame([{
    "Yöntem": m,
    "Parametre": count_params(models[m]) if m in models else 0,
    "RMSE tüm (dB)": np.sqrt(np.mean((p - y_test) ** 2)),
    "RMSE ≤ -4 dB": np.sqrt(np.mean((p[low] - y_test[low]) ** 2)),
    "RMSE > -4 dB": np.sqrt(np.mean((p[~low] - y_test[~low]) ** 2)),
    "ms / frame": infer_ms[m],
} for m, p in preds.items()])

os.makedirs(config.TABLES_DIR, exist_ok=True)
summary.to_csv(f"{config.TABLES_DIR}/summary_N{N}.csv", index=False)
pd.concat({m: d.set_index("snr_db") for m, d in results.items()}, axis=1) \
  .to_csv(f"{config.TABLES_DIR}/per_snr_N{N}.csv")
summary.round(4)

---
## 6. Figürleri ve tabloları indir

Colab kapanınca dosyalar silinir. Buradan zip olarak indir.

In [ ]:
!zip -qr /content/results.zip results
from google.colab import files
files.download("/content/results.zip")